# IAM을 사용한 아이덴티티 격리 기반 Amazon Bedrock AgentCore 런타임 및 AgentCore 메모리 에이전트

## 개요

이 튜토리얼은 IAM 기반 아이덴티티 전파를 활용하여 Amazon Bedrock AgentCore 메모리에서 안전한 메모리 격리를 구현하는 방법을 보여줍니다. 인증된 사용자 자격 증명을 기반으로 대화 기록을 자동으로 분할하고 격리하는 메모리 지원 에이전트를 구축하여, 멀티 테넌트 환경에서 데이터 프라이버시와 보안을 보장합니다.

메모리 격리는 프로덕션 대화형 에이전트의 핵심 보안 요구 사항입니다. 적절한 격리가 없으면 사용자가 다른 사용자의 대화 기록에 접근할 수 있어 프라이버시 위반과 데이터 유출이 발생할 수 있습니다. 이 튜토리얼은 IAM 정책을 활용하여 인증된 사용자 자격 증명에 기반한 안전한 격리된 메모리 공간을 생성하는 데 중점을 둡니다.

이 구현은 AgentCore 메모리가 AWS IAM과 통합되어 인증된 사용자 자격 증명에 기반한 메모리 경계를 자동으로 적용하는 방법을 보여줍니다.

### 튜토리얼 세부 정보

| 항목                | 세부 내용                                                        |
|---------------------|------------------------------------------------------------------|
| 튜토리얼 유형       | 보안 및 아이덴티티 관리                                          |
| 에이전트 유형       | 단일 대화형 에이전트                                             |
| 에이전트 프레임워크 | Strands 에이전트                                                 |
| LLM 모델            | Anthropic Claude Haiku 4.5                                      |
| 주요 기능           | 메모리 격리, IAM, 사용자 컨텍스트                                |
| 사용 SDK            | boto3, bedrock-agentcore, bedrock-agentcore-starter-toolkit      |

### 학습 내용

이 튜토리얼에서 배울 내용:
1. AgentCore 메모리에서 메모리 격리를 위한 IAM 정책 구성 방법
2. 에이전트 호출을 통한 사용자 아이덴티티 전파 방법
3. 격리된 메모리 공간을 가진 다중 사용자 에이전트 배포 및 테스트 방법
4. IAM 정책을 사용하여 다른 사용자 간 메모리 격리 검증 방법


### 아키텍처

이 예제는 IAM 기반 메모리 격리가 포함된 AgentCore 런타임에 배포된 대화형 에이전트를 보여줍니다:

<div style="text-align:left">
    <img src="architecture.png" width="90%"/>
</div>

## 0. 사전 요구사항

이 튜토리얼을 실행하려면 다음이 필요합니다:
* Python 3.10 이상
* Bedrock, ECR, IAM 및 Cognito에 대한 적절한 권한이 구성된 AWS 자격 증명
* Amazon Bedrock AgentCore SDK 및 종속성

먼저 필요한 라이브러리를 설치하겠습니다.

In [ ]:
!pip install -qUr requirements.txt

### 환경 설정

필요한 라이브러리를 가져오고 환경을 구성하겠습니다. 다음을 사용합니다:
- AWS 서비스 상호작용을 위한 `boto3`
- 에이전트 메모리 관리를 위한 `bedrock_agentcore.memory`
- 인증 설정을 위한 다양한 유틸리티 함수

In [ ]:
# 가져오기
import os
import uuid
import logging
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from utils import setup_cognito_user_pool, reauthenticate_users, get_user_sub, create_agentcore_role

# 구성
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")
REGION = os.getenv('AWS_REGION', 'us-west-2')

## 1. Amazon Cognito 사용자 풀 생성

이 섹션에서는 Amazon Cognito 사용자 풀과 사용자를 생성합니다. Cognito는 에이전트에 대한 사용자 인증 및 아이덴티티 관리를 제공하여, 각 사용자의 대화 기록이 해당 사용자만 접근할 수 있도록 보장합니다.

`setup_cognito_user_pool` 함수는 다음을 수행합니다:
1. Cognito 사용자 풀이 없으면 생성
2. 인증을 위한 앱 클라이언트 설정
3. 임시 비밀번호로 2명의 테스트 사용자 생성
4. 테스트를 위한 접근 토큰 생성

In [ ]:
print("Amazon Cognito 사용자 풀 및 사용자 설정 중...")
cognito_config = setup_cognito_user_pool(region=REGION)
print("Cognito 설정 완료")

## 2. 메모리 리소스 생성

이 섹션에서는 에이전트의 대화 기록을 저장할 메모리 리소스를 생성합니다. 메모리를 통해 에이전트는 과거 상호작용을 회상하고, 맥락을 유지하며, 시간이 지남에 따라 더 일관된 응답을 제공할 수 있습니다.

이 예제에서는 추가 장기 전략 없이 간단한 단기 메모리 리소스를 생성합니다. 메모리는 모든 대화 메시지를 저장하여, AgentCore 런타임에서 세션이 종료된 후에도 이전 상호작용을 기억할 수 있도록 도와줍니다.

In [ ]:
# 이 리소스의 고유 식별자 생성
unique_id = str(uuid.uuid4())[:8]
memory_name = f"RuntimeIdentityMemoryAgent_{unique_id}"

# 메모리 매니저 초기화
memory_manager = MemoryManager(region_name=REGION)

# 메모리 생성
print("\n메모리 생성 중...")
print("   2-3분 소요됩니다...\n")

memory = memory_manager.get_or_create_memory(
    name=memory_name,
    strategies=[],
    description="Memory isolation with IAM example.",
    event_expiry_days=30  # 선택 사항: 필요에 따라 조정
)

MEMORY_ID = memory.get("id")
print(f"\n메모리가 성공적으로 생성되었습니다!")
print(f"   메모리 ID: {MEMORY_ID}")
print(f"   상태: {memory.get('status')}")

## 3. 메모리 지원 에이전트 생성

이 섹션에서는 메모리 통합을 위한 커스텀 훅이 포함된 Strands 에이전트 프레임워크를 사용하여 메모리 지원 에이전트를 구축합니다. 이 에이전트는 AgentCore 메모리에서 메시지를 저장하고 검색하여 대화 맥락을 유지합니다.

> **메모리가 중요한 이유**: AgentCore 런타임의 세션은 일정 시간 후 만료되어 대화 맥락이 삭제됩니다. 대화를 메모리에 저장하면 세션 간에도 이전 정보가 유지되어, 긴 중단 후에도 사용자에게 원활한 경험을 제공합니다.

### 에이전트 기능

에이전트는 다음을 수행합니다:
1. 각 사용자 및 어시스턴트 메시지를 메모리에 자동 저장
2. 기존 세션 계속 시 과거 대화 기록 검색
3. 동일 사용자와의 여러 상호작용에 걸쳐 맥락 유지
4. 사용자 아이덴티티 확인을 통한 다른 사용자 간 대화 격리

### 구현의 핵심 구성 요소

#### 1. 메모리 훅 프로바이더
커스텀 훅 프로바이더가 구현하는 내용:
- `on_agent_initialized`: 에이전트 시작 시 트리거되어 AgentCore 메모리에서 대화 기록 검색
- `on_message_added`: 새 메시지가 대화에 추가될 때 트리거되어 AgentCore 메모리에 저장

#### 2. 에이전트 초기화
`initialize_agent` 함수:
- 올바른 리전으로 메모리 훅 구성
- 적절한 상태 변수(memory_id, actor_id, session_id)로 에이전트 설정
- LLM의 시스템 프롬프트 구성

#### 3. 사용자 검증
`get_user_sub` 함수:
- JWKS에 대해 Cognito 접근 토큰을 검증하고 사용자의 sub(고유 ID)를 반환

#### 4. 진입점 핸들러
runtime_memory_agent 함수:
- 입력 페이로드 파싱 및 사용자 메시지 추출
- Cognito의 JWT 토큰을 사용한 사용자 아이덴티티 검증
- 에이전트 초기화 및 세션 추적 관리
- 적절한 맥락으로 에이전트 호출 처리
- 런타임 환경에 형식화된 응답 반환

에이전트 파일을 생성하겠습니다:

In [ ]:
%%writefile runtime_identity_memory_agent.py
import os
import jwt
import ast
import json
import logging
from strands import Agent
from jwt import PyJWKClient
from typing import Dict, Any
from strands.models import BedrockModel
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent
from bedrock_agentcore.memory.constants import StrategyType, ConversationalMessage, MessageRole

# 상세 로깅 구성
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")

# AgentCore 앱 초기화
app = BedrockAgentCoreApp()

MODEL_ID = os.getenv('MODEL_ID')
MEMORY_ID = os.getenv('MEMORY_ID')
COGNITO_USER_POOL = os.getenv('COGNITO_USER_POOL')
REGION = os.getenv('AWS_REGION')

# 글로벌 에이전트 인스턴스 - 첫 번째 요청 시 초기화됨
agent = None

class MemoryHookProvider(HookProvider):

    def __init__(self):
        logger.info(f"MemoryHookProvider 초기화 중")
        self.memory_session_manager = MemorySessionManager(MEMORY_ID, REGION)
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트 시작 시 최근 대화 기록 로드"""
        logger.info("에이전트 초기화 훅 트리거됨")
        
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")
        
        logger.info(f"상태 값 - actor_id: {actor_id}, session_id: {session_id}")
        
        if not all([actor_id, session_id]):
            logger.warning("필수 상태 값이 없습니다")
            return
        
        try:
            # 세션이 존재하는지 확인
            logger.info(f"세션 {session_id} 존재 여부 확인 중...")
            session_exists = False
            try:
                events = self.memory_session_manager.list_events(
                    actor_id=actor_id,
                    session_id=session_id,
                    max_results=1
                )
                session_exists = len(events) > 0
                logger.info(f"세션 존재: {session_exists}")
            except Exception as e:
                logger.warning(f"세션 존재 확인 오류: {e}")
                session_exists = False
            
            if not session_exists:
                logger.info(f"세션 {session_id}에 대한 기존 대화를 찾을 수 없습니다")
                return
            
            # 대화 기록 로드
            logger.info(f"기존 세션 {session_id}의 대화 기록 로드 중")
            recent_turns = self.memory_session_manager.get_last_k_turns(
                actor_id=actor_id,
                session_id=session_id,
                k=5
            )            

            if recent_turns:
                logger.info(f"메모리에서 {len(recent_turns)}개의 대화 턴 로드 완료")
                
                # 에이전트의 대화 기록에 메시지 추가
                for turn in reversed(recent_turns):
                    for message in turn:
                        role = message['role'].lower()  # 'user' 또는 'assistant'
                        parsed = ast.literal_eval(message['content']['text'])
                        content = parsed[0]['text']
                        
                        # 에이전트의 메시지 기록에 추가
                        event.agent.messages.append({
                            "role": role,
                            "content": [{"text": content}]
                        })
                        logger.info(f"{role} 메시지를 기록에 추가: {content[:50]}...")
                
                logger.info(f"대화 기록에 {len(event.agent.messages)}개 메시지 추가 완료")
            else:
                logger.info("이 세션에 대한 최근 턴이 없습니다")
                    
        except Exception as e:
            logger.error(f"메모리 로드 오류: {e}", exc_info=True)
    
    def on_message_added(self, event: MessageAddedEvent):
        """메모리에 메시지 저장"""
        logger.info("메시지 추가됨 - 메모리에 저장 중")
        
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")
        
        if not all([actor_id, session_id]):
            logger.warning("필수 상태 값이 없습니다")
            return
        
        try:
            messages = event.agent.messages
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            if last_message["role"] == "user":
                message_role = MessageRole.USER
            elif last_message["role"] == "assistant":
                message_role = MessageRole.ASSISTANT
            
            self.memory_session_manager.add_turns(
                actor_id=actor_id,
                session_id=session_id,
                messages=[ConversationalMessage(message_content, message_role)]
            )
            logger.info("메시지 저장 완료")
            
        except Exception as e:
            logger.error(f"메시지 저장 오류: {e}")
    
    def register_hooks(self, registry: HookRegistry):
        logger.info("메모리 훅 등록 중")
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

def initialize_agent(actor_id, session_id):
    """첫 사용을 위한 에이전트 초기화"""
    global agent
    
    logger.info(f"actor_id={actor_id}, session_id={session_id}로 에이전트 초기화 중")
    
    # 모델 및 메모리 훅 생성
    logger.info(f"모델 ID 설정: {MODEL_ID}")
    model = BedrockModel(model_id=MODEL_ID)
    logger.info(f"메모리 훅 생성 중")
    memory_hook = MemoryHookProvider()
    
    # 적절한 초기 상태로 에이전트 생성
    logger.info("메모리 훅으로 에이전트 생성 중")
    agent = Agent(
        model=model,
        hooks=[memory_hook],
        system_prompt="You're a helpful, memory-enabled agent deployed on AgentCore Runtime. You can remember previous interactions within the same session. Be friendly and concise in your responses.",
        state={
            "actor_id": actor_id,
            "session_id": session_id
        }
    )
    logger.info(f"에이전트 초기화 완료. 상태: {agent.state.get()}")

def get_user_sub(access_token: str, region: str, user_pool_id: str) -> str:
    """
    Verifies a Cognito access token against JWKS and returns the user's sub (unique ID).

    :param access_token: The JWT access token string
    :param region: AWS region of the Cognito User Pool
    :param user_pool_id: The Cognito User Pool ID
    :return: The user's 'sub' claim if the token is valid
    :raises jwt.InvalidTokenError: If verification fails
    """
    access_token = access_token[7:]
    jwks_url = f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}/.well-known/jwks.json"
    jwks_client = PyJWKClient(jwks_url)
    signing_key = jwks_client.get_signing_key_from_jwt(access_token)

    decoded = jwt.decode(
        access_token,
        signing_key.key,
        algorithms=["RS256"],
        issuer=f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}",
        options={"require": ["exp", "iat", "iss", "token_use"]}
    )

    if decoded.get("token_use") != "access":
        raise jwt.InvalidTokenError("Token is not an access token")

    return decoded["sub"]

@app.entrypoint
def runtime_memory_agent(payload, context):
    """
    Main entry point for the memory-enabled agent
    
    Args:
        payload: The input payload containing user data
        context: The runtime context object containing session information
    """
    global agent
    
    # 페이로드 및 컨텍스트 정보 로깅
    logger.info(f"수신된 페이로드: {payload}")
    logger.info(f"컨텍스트: {context}")
    logger.info(f"사용자 Sub: {get_user_sub(context.request_headers.get('Authorization'), REGION, COGNITO_USER_POOL)}")
    
    # 필수 값 추출 및 검증
    user_input = payload.get("prompt")
    actor_id = get_user_sub(context.request_headers.get('Authorization'), REGION, COGNITO_USER_POOL)
    session_id = context.session_id  # 컨텍스트에서 session_id 가져오기
    
    # 필수 필드 검증
    if user_input is None:
        error_msg = "오류: 페이로드에 'prompt' 필드가 없습니다"
        logger.error(error_msg)
        return error_msg
    
    # 첫 번째 요청 시 에이전트 초기화
    if agent is None:
        logger.info("첫 번째 요청 - 에이전트 초기화 중")
        initialize_agent(actor_id, session_id)
    else:
        logger.info("기존 에이전트 인스턴스 사용 중")
        # 세션 ID가 변경된 경우 업데이트
        if agent.state.get("session_id") != session_id:
            logger.info(f"세션 ID를 {session_id}로 업데이트")
            agent.state.set("session_id", session_id)
        if agent.state.get("actor_id") != actor_id:
            logger.info(f"액터 ID를 {actor_id}로 업데이트")
            agent.state.set("actor_id", actor_id)
    
    logger.info(f"에이전트 시스템 프롬프트: {agent.system_prompt}")
    # 사용자 입력으로 에이전트 호출
    logger.info(f"에이전트 호출 입력: {user_input}")
    response = agent(user_input)
    response_text = response.message['content'][0]['text']
    logger.info(f"에이전트 응답: {response_text[:50]}...")
    
    return response_text

if __name__ == "__main__":
    logger.info("AgentCore 애플리케이션 시작 중")
    app.run()

## 4. AgentCore 런타임에 배포

이 섹션에서는 확장성과 간소화된 운영을 제공하는 관리형 에이전트 런타임 환경인 Amazon Bedrock AgentCore 런타임에 에이전트를 배포합니다. AgentCore 런타임은 인프라 복잡성을 처리하여, 배포 관리보다 에이전트 로직에 집중할 수 있게 합니다.

수동 서버 설정 및 관리가 필요한 기존 배포 방법과 달리, AgentCore 런타임은 에이전트 컨테이너를 AWS 인프라에 배포하고 호출을 위한 보안 HTTPS 엔드포인트를 제공합니다. 이 접근 방식은 에이전트가 수요에 따라 확장하고 프로덕션 환경에서 안정적으로 운영될 수 있도록 보장합니다.

> **팁**: AgentCore 스타터 툴킷이 IAM 역할, ECR 저장소, 컨테이너 빌드를 포함한 모든 복잡한 배포 단계를 처리합니다.

### 배포 구성

배포 구성을 설정하겠습니다:

In [ ]:
iam_role = create_agentcore_role(agent_name=f"runtime_memory_agent_{unique_id}", region=REGION)

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
import time

agentcore_runtime = Runtime()
agent_name = f"runtime_memory_agent_{unique_id}"

response = agentcore_runtime.configure(
    entrypoint="runtime_identity_memory_agent.py", 
    execution_role = iam_role["Role"]["RoleName"],
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=agent_name,
    non_interactive=True, 
    memory_mode="NO_MEMORY",
    idle_timeout = 60,
    request_header_configuration = {"requestHeaderAllowlist": ["Authorization"]},
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": cognito_config.get("discovery_url"),
            "allowedClients": [cognito_config.get("client_id")]
        }
    }
)
response

### 에이전트 시작

이제 AgentCore 런타임에 에이전트를 시작합니다. 이 단계에서는 구성된 에이전트를 AgentCore의 관리형 인프라에 배포합니다. 이 과정에서 에이전트에 필요한 필수 환경 변수를 전달합니다: 앞서 생성한 메모리 ID, 사용할 모델 ID, AWS 리전, 인증을 위한 Cognito 사용자 풀 ID입니다.

배포 후 에이전트는 사용자 메시지로 호출할 수 있는 보안 엔드포인트를 통해 접근 가능합니다. 엔드포인트는 Cognito 인증으로 보호되어 인가된 사용자만 에이전트에 접근할 수 있습니다.

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        "MEMORY_ID": MEMORY_ID,
        "MODEL_ID": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
        "AWS_REGION": REGION,
        "COGNITO_USER_POOL": cognito_config["pool_id"]
    }
)

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"현재 상태: {status}")

if status == 'READY':
    print("에이전트 배포 성공!")
else:
    print(f"배포 종료 상태: {status}")

account_id = status_response.config.account
role_arn = status_response.config.execution_role
role_name = role_arn.split("/")[-1]
print(f"계정 ID: {account_id}")
print(f"에이전트 역할: {role_arn}")
print(f"역할 이름: {role_name}")

## 5. 에이전트 테스트

에이전트가 배포되었으므로, 메시지를 보내고 이전 상호작용을 기억하는지 확인하여 테스트하겠습니다. 또한 다른 사용자가 격리된 메모리 컨텍스트를 갖는지 테스트하여, 한 사용자의 대화가 다른 사용자에게 보이지 않도록 합니다.

**세션 관리에 대한 중요 참고사항**

- **세션 관리**: AgentCore 런타임은 세션 ID가 제공되지 않으면 자동으로 생성하지만, 애플리케이션에서 세션 ID를 명시적으로 관리하는 것이 권장됩니다. 이를 통해 다음을 더 잘 제어할 수 있습니다:
  - 세션 타임아웃 후 대화 계속
  - 적절한 시점에 새 세션 생성 (예: 사용자가 새 대화를 시작할 때)
  - 동일 사용자와의 여러 병렬 대화 처리
  - 애플리케이션 요구에 따른 세션 만료 정책 구현

- **메모리 지속성**: AgentCore 런타임에서 세션이 만료되더라도, 동일 사용자로 새 세션이 시작될 때 에이전트는 AgentCore 메모리에서 이전 대화를 검색할 수 있습니다.

In [ ]:
# 필요한 경우 재인증
tokens = reauthenticate_users(cognito_config["client_id"], REGION)
cognito_config["bearer_tokens"]["testuser1"] = tokens["testuser1"]
cognito_config["bearer_tokens"]["testuser2"] = tokens["testuser2"]

In [ ]:
import time
import boto3
import json

def test_user_memory_isolation_with_iam():
    """
    Cognito 주체 ID를 사용한 동적 IAM 정책 제어로 사용자 메모리 격리를 테스트합니다.
    """
    print("\n" + "=" * 50)
    print("IAM 정책을 활용한 사용자 메모리 격리 테스트")
    print("=" * 50)
    
    # IAM 클라이언트 설정
    iam_client = boto3.client('iam')
    
    # 베어러 토큰 추출
    testuser1_token = cognito_config["bearer_tokens"]["testuser1"]
    testuser2_token = cognito_config["bearer_tokens"]["testuser2"]
    
    user_pool_id = cognito_config["pool_id"]
    
    # Cognito sub 가져오기 (이것이 액터 ID가 됨)
    testuser1_actor_id = get_user_sub(testuser1_token, REGION, user_pool_id)
    testuser2_actor_id = get_user_sub(testuser2_token, REGION, user_pool_id)
    
    print(f"TestUser1 Cognito Sub (액터 ID): {testuser1_actor_id}")
    print(f"TestUser2 Cognito Sub (액터 ID): {testuser2_actor_id}")
    
    # 동적 정책 템플릿 정의
    def create_actor_policy(actor_id, policy_name):
        policy_document = {
            "Version": "2012-10-17",
            "Statement": [
                {
                    "Sid": "DynamicActorIdRestriction",
                    "Effect": "Allow",
                    "Action": [
                        "bedrock-agentcore:CreateEvent",
                        "bedrock-agentcore:GetEvent",
                        "bedrock-agentcore:GetMemory",
                        "bedrock-agentcore:GetMemoryRecord",
                        "bedrock-agentcore:ListActors",
                        "bedrock-agentcore:ListEvents",
                        "bedrock-agentcore:ListMemoryRecords",
                        "bedrock-agentcore:ListSessions",
                        "bedrock-agentcore:DeleteEvent",
                        "bedrock-agentcore:DeleteMemoryRecord",
                        "bedrock-agentcore:RetrieveMemoryRecords"
                    ],
                    "Resource": [
                        f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:memory/*"
                    ],
                    "Condition": {
                        "StringEquals": {
                            "bedrock-agentcore:actorId": actor_id
                        }
                    }
                }
            ]
        }
        
        try:
            # 정책 생성 또는 업데이트
            try:
                response = iam_client.create_policy(
                    PolicyName=policy_name,
                    PolicyDocument=json.dumps(policy_document),
                    Description=f"Dynamic actor restriction policy for {actor_id}"
                )
                policy_arn = response['Policy']['Arn']
            except iam_client.exceptions.EntityAlreadyExistsException:
                # 기존 정책 업데이트
                policy_arn = f"arn:aws:iam::{account_id}:policy/{policy_name}"
                iam_client.create_policy_version(
                    PolicyArn=policy_arn,
                    PolicyDocument=json.dumps(policy_document),
                    SetAsDefault=True
                )
                
            # 역할에 정책 연결
            iam_client.attach_role_policy(
                RoleName=role_name,
                PolicyArn=policy_arn
            )
            
            print(f"Cognito sub (액터 ID)에 대한 정책 업데이트: {actor_id}")
            return policy_arn
            
        except Exception as e:
            print(f"정책 업데이트 오류: {e}")
            raise
    
    # 정책 분리 함수 정의
    def detach_policy(policy_arn):
        try:
            iam_client.detach_role_policy(
                RoleName=role_name,
                PolicyArn=policy_arn
            )
            print(f"정책 {policy_arn} 분리 완료")
        except Exception as e:
            print(f"정책 분리 오류: {e}")
    
    # 각 테스트 단계별 고유 세션 ID 생성
    user1_session_id = f"memory-agent-session-user1-{int(time.time())}"
    user2_session_id = f"memory-agent-session-user2-{int(time.time())}"
    
    
    # 이 테스트를 위한 정책 생성
    policy_name = f"actor-restriction-{int(time.time())}"
    
    try:
        # 단계 1: 올바른 정책으로 user1의 메모리 지속성 테스트
        print("\n" + "=" * 50)
        print("단계 1: 사용자 1 메모리 지속성 테스트")
        print("=" * 50)

        # user 1을 위한 정책 생성
        policy_arn = create_actor_policy(testuser1_actor_id, policy_name)

        # 정책 전파 대기
        print("IAM 정책 전파 대기 중...")
        time.sleep(10)

        # 단계 1: 사용자 1이 초기 정보 공유
        print("\n" + "-" * 50)
        print("단계 1: 사용자 1이 정보를 공유합니다")
        print("-" * 50)

        # "내 이름은 John이고 좋아하는 색상은 보라색입니다"
        user1_prompt1 = "My name is John and my favorite color is purple."
        response1 = agentcore_runtime.invoke(
            {"prompt": user1_prompt1},
            session_id=user1_session_id,
            bearer_token=testuser1_token
        )
        print(f"사용자 1 프롬프트: \"{user1_prompt1}\"")
        print(f"사용자 1 응답: \"{response1['response']}\"")

        # 세션 종료 대기 (75초)
        print("\n세션 종료를 위해 75초 대기 중...")
        time.sleep(75)

        # 단계 2: 사용자 1이 정보 회상 요청 (정책으로 작동해야 함)
        print("\n" + "-" * 50)
        print("단계 2: 사용자 1이 정보를 회상합니다 (성공해야 함)")
        print("-" * 50)

        # "내 이름과 좋아하는 색상이 뭔가요?"
        user1_prompt2 = "What is my name and favorite color?"
        response2 = agentcore_runtime.invoke(
            {"prompt": user1_prompt2},
            session_id=user1_session_id,  
            bearer_token=testuser1_token
        )
        print(f"사용자 1 프롬프트: \"{user1_prompt2}\"")
        print(f"사용자 1 응답: \"{response2['response']}\"")

        # 단계 2: user2 메모리 격리 테스트
        print("\n" + "=" * 50)
        print("단계 2: 사용자 2 메모리 격리 테스트")
        print("=" * 50)

        # 단계 3: 사용자 2가 정보 공유 (메모리 생성되지만 API 호출로 작동)
        print("\n" + "-" * 50)
        print("단계 3: 사용자 2가 정보를 공유합니다")
        print("-" * 50)

        # 초기 상호작용을 허용하기 위해 임시로 정책을 user 2로 변경
        detach_policy(policy_arn)
        policy_arn = create_actor_policy(testuser2_actor_id, policy_name)

        # 정책 업데이트 전파 대기
        print("업데이트된 IAM 정책 전파 대기 중...")
        time.sleep(10)

        # "내 이름은 Mary이고 좋아하는 음식은 파스타입니다"
        user2_prompt1 = "My name is Mary and my favorite food is pasta."
        response3 = agentcore_runtime.invoke(
            {"prompt": user2_prompt1},
            session_id=user2_session_id,
            bearer_token=testuser2_token
        )
        print(f"사용자 2 프롬프트: \"{user2_prompt1}\"")
        print(f"사용자 2 응답: \"{response3['response']}\"")

        # 세션 종료 대기
        print("\n세션 종료를 위해 75초 대기 중...")
        time.sleep(75)

        # 단계 4: 정책을 user 1로 되돌린 후, user 2가 회상 시도 (실패해야 함)
        print("\n" + "-" * 50)
        print("단계 4: 정책을 사용자 1로 변경, 사용자 2가 회상 시도 (실패해야 함)")
        print("-" * 50)

        # 정책을 user 1로 되돌리기
        detach_policy(policy_arn)
        policy_arn = create_actor_policy(testuser1_actor_id, policy_name)

        # 정책 업데이트 전파 대기
        print("업데이트된 IAM 정책 전파 대기 중...")
        time.sleep(10)

        # "내 이름과 좋아하는 음식이 뭔가요?"
        user2_prompt2 = "What is my name and favorite food?"
        response4 = agentcore_runtime.invoke(
            {"prompt": user2_prompt2},
            session_id=user2_session_id, 
            bearer_token=testuser2_token
        )
        print(f"사용자 2 프롬프트: \"{user2_prompt2}\"")
        print(f"사용자 2 응답: \"{response4['response']}\"")
        print("\n 정책이 사용자 1 이벤트에만 접근 권한을 부여하므로 에이전트는 사용자 2 메모리에 접근할 수 없어야 합니다")
        
    finally:
        # 정리 - 정책 제거
        detach_policy(policy_arn)

In [ ]:
test_user_memory_isolation_with_iam()

## 핵심 개념

이 튜토리얼에서 Bedrock AgentCore로 메모리 지원 에이전트를 구축하기 위한 몇 가지 중요한 개념을 배웠습니다:

1. **메모리 통합**: AgentCore 메모리를 사용하여 세션 간 대화 기록을 저장하는 방법으로, 세션이 만료되더라도 에이전트가 시간이 지남에 따라 맥락을 유지할 수 있게 합니다.

2. **세션 관리**: 세션 ID를 사용하여 대화를 구성하고 사용자가 돌아올 때 관련 기록을 검색하여 원활한 경험을 만드는 방법입니다.

3. **AgentCore 배포**: 확장, 보안 및 인프라 관리를 자동으로 처리하는 프로덕션 런타임 환경에 에이전트를 배포하는 방법입니다.

4. **메모리 훅**: 메모리 서비스와 통합하는 커스텀 훅을 구현하여, 에이전트 라이프사이클의 특정 시점에서 대화 기록을 저장하고 검색하는 방법입니다.

5. **사용자 아이덴티티 및 프라이버시**: 인증을 사용하여 각 사용자의 대화 기록이 비공개이며 다른 사용자로부터 격리되도록 보장하는 방법입니다.

이러한 개념은 지속적인 메모리와 정교한 대화 관리 기능을 갖춘 더 복잡한 에이전트를 구축하기 위한 기반을 제공합니다.

## 정리 (선택 사항)

이 튜토리얼에서 생성한 리소스가 더 이상 필요하지 않다면, 불필요한 AWS 비용을 방지하기 위해 정리할 수 있습니다.

In [ ]:
# 모든 리소스를 삭제하려면 이 셀을 실행하세요

# 1. AgentCore 런타임 삭제
if 'launch_result' in locals() and hasattr(launch_result, 'agent_id'):
    try:
        agentcore_control_client = boto3.client(
            'bedrock-agentcore-control',
            region_name=REGION
        )
        
        runtime_delete_response = agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=launch_result.agent_id,
        )
        print(f"AgentCore 런타임 삭제 완료: {launch_result.agent_id}")
    except Exception as e:
        print(f"AgentCore 런타임 삭제 오류: {e}")
else:
    print("삭제할 AgentCore 런타임이 없습니다")

# 2. ECR 저장소 삭제
if 'launch_result' in locals() and hasattr(launch_result, 'ecr_uri'):
    try:
        ecr_client = boto3.client(
            'ecr',
            region_name=REGION
        )
        
        repository_name = launch_result.ecr_uri.split('/')[1]
        response = ecr_client.delete_repository(
            repositoryName=repository_name,
            force=True  # 이미지가 있어도 강제 삭제
        )
        print(f"ECR 저장소 삭제 완료: {repository_name}")
    except Exception as e:
        print(f"ECR 저장소 삭제 오류: {e}")
else:
    print("삭제할 ECR 저장소가 없습니다")

# 3. 메모리 리소스 삭제
try:
    memory_manager.delete_memory(memory_id=MEMORY_ID)
    print(f"메모리 리소스 삭제 완료: {MEMORY_ID}")
except Exception as e:
    print(f"메모리 리소스 삭제 오류: {e}")

# 4. Cognito 사용자 풀 및 관련 리소스 삭제
if 'cognito_config' in locals() and cognito_config and 'pool_id' in cognito_config:
    try:
        cognito_client = boto3.client('cognito-idp', region_name=REGION)
        
        # 사용자 풀 ID 가져오기
        pool_id = cognito_config['pool_id']
        
        # 모든 사용자 풀 클라이언트 나열 및 삭제
        clients_response = cognito_client.list_user_pool_clients(
            UserPoolId=pool_id,
            MaxResults=60
        )
        
        for client in clients_response.get('UserPoolClients', []):
            client_id = client['ClientId']
            cognito_client.delete_user_pool_client(
                UserPoolId=pool_id,
                ClientId=client_id
            )
            print(f"사용자 풀 클라이언트 삭제 완료: {client_id}")
        
        # 사용자 풀 자체 삭제
        cognito_client.delete_user_pool(
            UserPoolId=pool_id
        )
        print(f"Cognito 사용자 풀 삭제 완료: {pool_id}")
        
    except Exception as e:
        print(f"Cognito 리소스 삭제 오류: {e}")
else:
    print("삭제할 Cognito 리소스가 없습니다")

# 5. IAM 역할 및 모든 버전 삭제 함수
def delete_iam_role(role_identifier, region=REGION):
    """
    연결된 모든 정책과 버전을 포함하여 IAM 역할을 삭제합니다
    
    Args:
        role_identifier (str): IAM 역할의 ARN 또는 이름
        region (str): AWS 리전
    """
    try:
        iam_client = boto3.client('iam', region_name=region)
        
        # 식별자가 ARN인지 역할 이름인지 확인
        if role_identifier.startswith('arn:aws:iam::'):
            # ARN에서 역할 이름 추출
            role_name = role_identifier.split('/')[-1]
        else:
            role_name = role_identifier
            
        print(f"IAM 역할 삭제 시도: {role_name}")
            
        # 1. 모든 관리형 정책 분리
        attached_policies = iam_client.list_attached_role_policies(RoleName=role_name)
        for policy in attached_policies.get('AttachedPolicies', []):
            iam_client.detach_role_policy(
                RoleName=role_name,
                PolicyArn=policy['PolicyArn']
            )
            print(f"관리형 정책 분리 완료: {policy['PolicyArn']}")
            
        # 2. 모든 인라인 정책 삭제
        inline_policies = iam_client.list_role_policies(RoleName=role_name)
        for policy_name in inline_policies.get('PolicyNames', []):
            iam_client.delete_role_policy(
                RoleName=role_name,
                PolicyName=policy_name
            )
            print(f"인라인 정책 삭제 완료: {policy_name}")
            
        # 3. 역할과 연결된 인스턴스 프로파일 삭제
        instance_profiles = iam_client.list_instance_profiles_for_role(RoleName=role_name)
        for profile in instance_profiles.get('InstanceProfiles', []):
            iam_client.remove_role_from_instance_profile(
                InstanceProfileName=profile['InstanceProfileName'],
                RoleName=role_name
            )
            print(f"인스턴스 프로파일에서 역할 제거 완료: {profile['InstanceProfileName']}")
            
        # 4. 최종적으로 역할 삭제
        iam_client.delete_role(RoleName=role_name)
        print(f"IAM 역할 삭제 성공: {role_name}")
        
    except Exception as e:
        print(f"IAM 역할 삭제 오류: {e}")


delete_iam_role(role_arn)

print("\n정리 완료")